In [1]:
import pandas as pd

def simular_hardware_mcu(neuronas_entrada, neuronas_salida, formato):
    """
    Simula el costo de memoria y latencia (en ciclos de reloj) de una capa Densa 
    basado en arquitecturas tipo STM32N6 (Cortex-M55 + NPU).
    """
    total_pesos = neuronas_entrada * neuronas_salida
    total_activaciones = neuronas_salida
    
    # Especificaciones reales aproximadas de Hardware
    bus_datos_bytes = 4 # Bus de 32 bits (trae 4 bytes por ciclo)
    
    # 1. Configuración según el formato de datos
    if formato == "Float32":
        bytes_por_peso = 4
        bytes_por_activacion = 4
        macs_por_ciclo = 1   # FPU estándar
        procesador = "CPU FPU"
        
    elif formato == "INT8 (Full PTQ/QAT)":
        bytes_por_peso = 1
        bytes_por_activacion = 1
        macs_por_ciclo = 32  # NPU Neural-ART (Altamente paralela)
        procesador = "NPU Neural-ART"
        
    elif formato == "INT16x8 (Mixed PTQ)":
        bytes_por_peso = 1
        bytes_por_activacion = 2
        macs_por_ciclo = 4   # Cortex-M55 con Helium (SIMD vectorial de 16-bit)
        procesador = "CPU + Helium Vectorial"
    else:
        return None

    # 2. Cálculos de Memoria
    memoria_flash_bytes = total_pesos * bytes_por_peso
    memoria_sram_bytes = total_activaciones * bytes_por_activacion
    
    # 3. Cálculos de Tráfico de Memoria (Cuello de botella)
    ciclos_para_leer_pesos = memoria_flash_bytes / bus_datos_bytes
    
    # 4. Cálculos de Cómputo
    total_macs = total_pesos
    ciclos_de_computo = total_macs / macs_por_ciclo
    
    # Latencia total (Lectura + Cómputo, asumiendo un pipeline simple)
    ciclos_totales = ciclos_para_leer_pesos + ciclos_de_computo
    
    return {
        "Formato": formato,
        "Motor de Hardware": procesador,
        "Flash Requerida (Bytes)": memoria_flash_bytes,
        "SRAM Requerida (Bytes)": memoria_sram_bytes,
        "Ciclos Lectura Bus": int(ciclos_para_leer_pesos),
        "Ciclos Cómputo (MAC)": int(ciclos_de_computo),
        "Ciclos Totales (Latencia)": int(ciclos_totales)
    }

# =========================================================
# EJECUCIÓN DE LA SIMULACIÓN
# =========================================================
# Simulamos una matriz podada de tu ejemplo anterior (16 x 16 neuronas)
N_IN = 16
N_OUT = 16

resultados = []
formatos = ["Float32", "INT8 (Full PTQ/QAT)", "INT16x8 (Mixed PTQ)"]

for f in formatos:
    resultados.append(simular_hardware_mcu(N_IN, N_OUT, f))

# Mostrar resultados como tabla (Pandas)
df_resultados = pd.DataFrame(resultados)
print(f"--- ANÁLISIS DE HARDWARE PARA CAPA DE {N_IN}x{N_OUT} ({N_IN*N_OUT} MACs) ---")
display(df_resultados) # Si usas Jupyter, usa 'display'. Si es consola, usa 'print'

--- ANÁLISIS DE HARDWARE PARA CAPA DE 16x16 (256 MACs) ---


,Formato,Motor de Hardware,Flash Requerida (Bytes),SRAM Requerida (Bytes),Ciclos Lectura Bus,Ciclos Cómputo (MAC),Ciclos Totales (Latencia)
0,Float32,CPU FPU,1024,64,256,256,512
1,INT8 (Full PTQ/QAT),NPU Neural-ART,256,16,64,8,72
2,INT16x8 (Mixed PTQ),CPU + Helium Vectorial,256,32,64,64,128
